In [1]:
import os
import sys
import uuid

from pathlib import Path
import numpy as np
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from modules.seismo_response.class_simulation import Nonlinear_Simulation, Equiv_Linear_Simulation
#from modules.seismo_response.class_parameters import GQH_Param_Multi_Layer
from modules.seismo_response.class_curves import(
    Multiple_GGmax_Damping_Curves
)
from modules.seismo_response.class_ground_motion import Ground_Motion
from modules.seismo_response.class_Vs_profile import Vs_Profile
from modules.seismo_response.class_parameters import GQH_Param_Multi_Layer, Param_Multi_Layer
from modules.seismo_response.class_parameters import Parameter, GQH_Param

from libs.config.config_variables import STORAGE_DIR, UNIT_GROUND_MOTION, STRAIN_RANGE_PCT


In [2]:
session = "sesion_20260223_103200_ew"

input_dir = STORAGE_DIR / "raw_data" / session
output_dir = STORAGE_DIR / "output_data" / session

In [6]:
accel_file = input_dir / "input_accel.txt"
vs_file = input_dir / "vs_profile.txt"
dynamic_curvas_file = input_dir / "dynamic_curves.txt"

input_accel = Ground_Motion(str(accel_file), unit=UNIT_GROUND_MOTION)
vs_profile = Vs_Profile(str(vs_file))

print(dynamic_curvas_file)
print(type(dynamic_curvas_file))

ggmax_and_damping_curves = Multiple_GGmax_Damping_Curves(data=str(dynamic_curvas_file))


c:\Users\william.ortiz\OneDrive - ANDDES ASOCIADOS SAC\Otros\Escritorio\Proyectos GITLAB\PRISMO_GITLAB\prismo\var\raw_data\sesion_20260223_103200_ew\dynamic_curves.txt
<class 'pathlib._local.WindowsPath'>


In [ ]:
print()

In [10]:
leq_sim = Equiv_Linear_Simulation(
    vs_profile,
    input_accel,
    ggmax_and_damping_curves,
    boundary='elastic',
    )

In [11]:
sim_results = leq_sim.run(
    show_fig=False,
    save_txt=False,
    )

2026-02-25 10:16:11,855 [INFO] modules.seismo_response.helper_simulations: ---------- Convergencia alcanzada ---------------


## No lineal

In [12]:
data = [
        {"gamma_ref": 0.022,
        "theta_1": 0.1, 
        "theta_2": 0.3, 
        "theta_3": 1.0, 
        "theta_4": 1.0, 
        "theta_5": 4.0,
        "Gmax": 1.0,
        },
        {"gamma_ref": 0.022,
        "theta_1": 0.1, 
        "theta_2": 0.3, 
        "theta_3": 1.0, 
        "theta_4": 1.0, 
        "theta_5": 4.0,
        "Gmax": 1.0,
        }
        ]
layers_GQH = Param_Multi_Layer(
    data,
    element_class=GQH_Param,
)

In [13]:
strain = STRAIN_RANGE_PCT

In [ ]:
# layer_1 = layers_GQH.param_list[0]
# layer_1


In [21]:
# tau = layers_GQH.get_GGmax()

In [22]:
mgc, mdc = layers_GQH.construct_curves()

tau type: <class 'numpy.ndarray'>
tau shape: (50,)
tau ndim: 1
tau type: <class 'numpy.ndarray'>
tau shape: (50,)
tau ndim: 1


In [ ]:
# strain_in_pct = np.linspace(0, 10, 100)
# GGmax = Parameter.get_GGmax(strain_in_pct=strain_in_pct)
# damping = Parameter.get_damping(strain_in_pct=strain_in_pct)

In [ ]:
nl_sim = Nonlinear_Simulation(
    vs_profile,
    input_accel,
    G_param=layers_GQH,
    xi_param=layers_GQH,
    boundary='elastic',
    )

In [ ]:
sim_results = nl_sim.run(
    show_fig=False,
    save_txt=False,
    remove_sim_dir=True,  # flip `save_txt` and `remove_sim_dir` to keep result as text files
    )